# Tutorial: Trace Direct vs Programmatic Tool Calling — Inventory Replenishment

**Audience:** Engineers evaluating OpenAI Responses API tool orchestration.

**Prerequisites:** Python, the Responses API tool-calling loop, and the difference between a model turn and a host-side tool execution.

**Learning goals:** By the end, you can inspect one comparison trace containing Direct and Programmatic sibling arms, follow observable tool-calling events, and compare quality, host round trips, payload size, tokens, latency, and estimated cost.

> Scope: traces expose observable API and tool lifecycle events. They do not expose hidden model reasoning.


## Outline

1. Configure safe live and Trace export controls.
2. Build the deterministic inventory fixture.
3. Inspect the shared tools and arm-specific orchestration.
4. Run one optional Direct/PTC comparison inside a parent OpenAI Trace.
5. Compare the semantic event timeline, quality, and resource metrics.
6. Practice interpreting host round trips and intermediate payloads.


## 1. Setup

The Trace adapter wraps the existing raw Responses API client and deterministic scenario. It records semantic events without replacing the benchmark's orchestration loop.


In [ ]:
from __future__ import annotations

import importlib.util
import os
import sys
import uuid
from pathlib import Path

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "ptc_benchmark").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ptc_benchmark").exists():
    raise RuntimeError("Run this notebook from the project root or notebooks directory.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ptc_benchmark.config import configured_model, load_local_environment, require_api_key
from ptc_benchmark.inventory import build_inventory_dataset
from ptc_benchmark.inventory_evaluation import evaluate_inventory_run
from ptc_benchmark.pricing import estimate_run_cost, load_pricing_catalog
from ptc_benchmark.reporting import markdown_table
from ptc_benchmark.runner import RunConfig
from ptc_trace_demo.inventory_trace import (
    configure_trace_error_logging,
    run_inventory_trace_comparison,
)


### Safe execution controls

Live API usage, OpenAI Trace export, and detailed trace-error logging use separate opt-ins. Full synthetic payloads remain local for timeline inspection; the OpenAI Trace backend receives bounded summaries only. Set `SHOW_TRACE_ERROR_DETAILS=True` temporarily to include the Trace ingest response body in errors. Detailed logs can contain sensitive model or tool data, so keep it `False` for normal use.


In [ ]:
RUN_LIVE = False # Set to True to enable API calls and associated cost.
EXPORT_OPENAI_TRACE = False # Set to True to export bounded summaries; requires RUN_LIVE = True.
SHOW_TRACE_ERROR_DETAILS = False # Set to True temporarily to show Trace ingest error bodies.
INCLUDE_LOCAL_PAYLOADS = True  # Retain full deterministic payloads in the local timeline only.

configure_trace_error_logging(show_details=SHOW_TRACE_ERROR_DETAILS)

SCALE = "small"  # small=3 SKUs, medium=10, large=30
MODEL = configured_model("gpt-5.6")
REASONING_EFFORT = "medium"
MAX_REQUESTS = 16  # Bounded continuation budget for both comparison arms.

PRICING_PATH = Path(
    os.getenv(
        "OPENAI_PRICING_PATH",
        PROJECT_ROOT / "pricing" / "openai_pricing_2026-08-14.json",
    )
)

print({
    "RUN_LIVE": RUN_LIVE,
    "EXPORT_OPENAI_TRACE": EXPORT_OPENAI_TRACE,
    "show_trace_error_details": SHOW_TRACE_ERROR_DETAILS,
    "include_local_payloads": INCLUDE_LOCAL_PAYLOADS,
    "scale": SCALE,
    "model": MODEL,
    "reasoning_effort": REASONING_EFFORT,
    "max_requests": MAX_REQUESTS,
})


## 2. Deterministic fixture and quality oracle

Both arms receive the same SKUs, tool schemas, model settings, and expected replenishment plan. Quality remains a hard gate before cost is interpreted.


In [ ]:
dataset = build_inventory_dataset(SCALE)
oracle = dataset.expected_plan()

print({
    "case_id": dataset.case_id,
    "sku_count": len(dataset.skus),
    "expected_tool_calls_per_arm": len(dataset.skus) * 3,
    "expected_recommendations": len(oracle["recommendations"]),
})


## 3. What the Trace should reveal

```
Inventory comparison trace
├── direct_arm
│   ├── model_request → function_call observations
│   ├── host tool executions and outputs
│   └── next model_request → final assistant message
└── programmatic_arm
    ├── model_request → program + function_call observations
    ├── hosted-program-linked tool executions and outputs
    └── next model_request → program_output + final assistant message
```

A `caller_id` on PTC function calls links each call to its generated program. Direct calls should have no program caller.


In [ ]:
contract_rows = []
prompt_sections = []
for arm in ("direct", "programmatic"):
    instructions, user_input = dataset.prompt(arm)
    tools = dataset.tool_definitions(arm)
    contract_rows.append({
        "arm": arm,
        "function_tools": sum(tool["type"] == "function" for tool in tools),
        "ptc_tool_enabled": any(tool["type"] == "programmatic_tool_calling" for tool in tools),
        "allowed_callers": sorted({
            caller
            for tool in tools
            if tool["type"] == "function"
            for caller in tool.get("allowed_callers", [])
        }),
        "user_prompt_chars": len(user_input),
    })
    prompt_sections.append(
        f"### {arm.title()}\n\n"
        f"**Instructions**\n\n```text\n{instructions}\n```\n\n"
        f"**User Prompt**\n\n```text\n{user_input}\n```"
    )

display(Markdown(markdown_table(contract_rows)))
display(Markdown("\n\n".join(prompt_sections)))


### Output example for Section 3: What the Trace should reveal

The following Markdown preserves an example of the output produced in this section.

| arm | function_tools | ptc_tool_enabled | allowed_callers | user_prompt_chars |
| --- | --- | --- | --- | --- |
| direct | 3 | False | ['direct'] | 67 |
| programmatic | 3 | True | ['programmatic'] | 67 |

### Direct

**Instructions**

```
<task_contract>
You are preparing a deterministic inventory replenishment plan as of 2026-08-10.

For every SKU in this exact list, call get_inventory, get_weekly_demand, and
get_inbound_shipments exactly once: sku-001, sku-002, sku-003.

For each SKU:
1. available_units = sum(on_hand_units - reserved_units) across warehouses.
2. forecast_units = sum(units) across all seven daily_forecast rows.
3. inbound_units = sum(units) only for shipments with status "scheduled" and
   eta_date on or before 2026-08-17.
4. reorder_units = max(forecast_units + 5 - available_units - inbound_units, 0).

Keep only positive reorder quantities. Sort by reorder_units descending and then
sku ascending. The structured result must be exactly one JSON object with this shape:
{"recommendations":[{"sku":"...","available_units":0,"forecast_units":0,
"inbound_units":0,"reorder_units":0}],"total_reorder_units":0}

The final assistant message must contain:
RESULT_JSON: <the exact one-line JSON object>
EXPLANATION: <a concise explanation that cites every recommended SKU and all four
source/calculated unit values for that SKU>.
Do not invent values and do not omit evidence from the explanation.
</task_contract>

<tool_orchestration>
Use Direct Tool Calling. Call the functions directly and issue independent calls in
parallel when possible. Use the returned tool data to calculate the result. Do not
write or execute a programmatic_tool_calling program.
</tool_orchestration>
```

**User Prompt**

```
Which products should we reorder this week, and in what quantities?
```

### Programmatic

**Instructions**

```
<task_contract>
You are preparing a deterministic inventory replenishment plan as of 2026-08-10.

For every SKU in this exact list, call get_inventory, get_weekly_demand, and
get_inbound_shipments exactly once: sku-001, sku-002, sku-003.

For each SKU:
1. available_units = sum(on_hand_units - reserved_units) across warehouses.
2. forecast_units = sum(units) across all seven daily_forecast rows.
3. inbound_units = sum(units) only for shipments with status "scheduled" and
   eta_date on or before 2026-08-17.
4. reorder_units = max(forecast_units + 5 - available_units - inbound_units, 0).

Keep only positive reorder quantities. Sort by reorder_units descending and then
sku ascending. The structured result must be exactly one JSON object with this shape:
{"recommendations":[{"sku":"...","available_units":0,"forecast_units":0,
"inbound_units":0,"reorder_units":0}],"total_reorder_units":0}

The final assistant message must contain:
RESULT_JSON: <the exact one-line JSON object>
EXPLANATION: <a concise explanation that cites every recommended SKU and all four
source/calculated unit values for that SKU>.
Do not invent values and do not omit evidence from the explanation.
</task_contract>

<tool_orchestration>
Use Programmatic Tool Calling for the complete lookup and calculation stage. Create
all tool-call promises before awaiting them and resolve them with Promise.all. Perform
all filtering, summation, sorting, and reduction inside the generated JavaScript.
Emit exactly the required JSON object from the program with text(JSON.stringify(result)).
After the program completes, write the required RESULT_JSON and EXPLANATION final message.
Do not call the inventory functions directly.
</tool_orchestration>
```

**User Prompt**

```
Which products should we reorder this week, and in what quantities?
```


## 4. Optional live comparison and OpenAI Trace export

Set `RUN_LIVE=True` for API execution. Also set `EXPORT_OPENAI_TRACE=True` to send bounded event summaries to the Trace backend. Full payloads stay in the local comparison object. The Trace dependency is included by `uv sync --group dev`.


In [ ]:
trace_comparison = None
trace_metric_rows = []

if not RUN_LIVE:
    print("Live comparison skipped. Set RUN_LIVE = True to opt in to API usage and cost.")
elif EXPORT_OPENAI_TRACE and importlib.util.find_spec("agents") is None:
    raise RuntimeError(
        "Trace export requires openai-agents: uv sync --group dev"
    )
else:
    from openai import OpenAI

    load_local_environment(PROJECT_ROOT)
    require_api_key()
    pricing = load_pricing_catalog(PRICING_PATH)
    comparison_id = f"inventory-trace-{uuid.uuid4().hex[:10]}"
    run_config = RunConfig(
        model=MODEL,
        reasoning_effort=REASONING_EFFORT,
        max_requests=MAX_REQUESTS,
    )
    trace_comparison = run_inventory_trace_comparison(
        client=OpenAI(),
        dataset=dataset,
        config=run_config,
        comparison_id=comparison_id,
        export_openai_trace=EXPORT_OPENAI_TRACE,
        include_payloads=INCLUDE_LOCAL_PAYLOADS,
    )

    intermediate_kinds = {"program", "function_call", "tool_output", "program_output"}
    for arm, run in trace_comparison.runs.items():
        evaluation = evaluate_inventory_run(run, dataset)
        cost = estimate_run_cost(run, pricing)
        arm_events = [event for event in trace_comparison.events if event.arm == arm]
        trace_metric_rows.append({
            "arm": arm,
            "quality_passed": evaluation.passed,
            "model_requests": len(run.requests),
            "host_round_trips": len(run.requests),
            "tool_calls": len(run.tool_calls),
            "intermediate_payload_bytes": sum(
                event.payload_bytes for event in arm_events if event.kind in intermediate_kinds
            ),
            "input_tokens": run.usage.input_tokens,
            "output_tokens": run.usage.output_tokens,
            "reasoning_tokens": run.usage.reasoning_output_tokens,
            "latency_seconds": round(run.total_latency_seconds, 3),
            "estimated_cost_usd": round(cost.total_cost, 6),
        })
        if not evaluation.passed:
            print(f"{arm} failed quality gates: {evaluation.failures}")

    display(Markdown(markdown_table(trace_metric_rows)))
    print({"comparison_id": trace_comparison.comparison_id, "trace_id": trace_comparison.trace_id})
    if trace_comparison.trace_id:
        display(Markdown("Open the [OpenAI Traces dashboard](https://platform.openai.com/traces) and search for the trace ID above."))


## 5. Inspect the normalized semantic timeline

The Notebook timeline is the reproducible companion to the Dashboard trace. It retains event order and comparison fields even when Trace export is disabled.


In [ ]:
if trace_comparison is None:
    print("No live timeline. Enable RUN_LIVE and rerun the comparison cell.")
else:
    timeline_columns = (
        "sequence", "arm", "elapsed_ms", "duration_ms", "event",
        "name", "request", "call_id", "caller_id", "payload_bytes",
    )
    timeline_rows = trace_comparison.timeline_rows()
    for arm, title in (("direct", "Direct"), ("programmatic", "Programmatic")):
        arm_timeline_rows = [row for row in timeline_rows if row["arm"] == arm]
        arm_rows = []
        for sequence, row in enumerate(arm_timeline_rows, start=1):
            display_row = {column: row[column] for column in timeline_columns}
            display_row["sequence"] = sequence
            arm_rows.append(display_row)
        display(Markdown(f"### {title} timeline\n\n{markdown_table(arm_rows)}"))


## 6. How to interpret the normalized timeline

### What the event order shows

- **Direct:** The first model request emits the inventory, demand, and inbound-shipment function calls. The host executes those calls, returns their tool outputs, and a second model request produces the final assistant message. Direct function calls have no program `caller_id`.
- **Programmatic:** The model first emits a `program`. Function calls made by that program share its `call_id` as their `caller_id`, which makes the parent-child relationship visible. After the program has gathered and reduced the tool results, a `program_output` is followed by the final assistant message.
- **Sequence numbers:** Each table starts at 1 for readability. They represent order within an arm, while `elapsed_ms` is measured from the beginning of the complete comparison, so the Programmatic table naturally starts after the Direct run.

### What this saved run indicates

#### Small dataset run

The interpretation below is based on the saved outputs from the `small` dataset run.

Reference outputs: live comparison trace export and normalized semantic timeline.

Both arms passed the quality gate and executed the same nine business-tool calls, so their resource measurements are comparable.

| measure | Direct | Programmatic | observation |
| --- | --- | --- | --- |
| Model requests / host round trips | 2 | 5 | Programmatic required three additional Responses API continuations in this run. |
| Intermediate payload bytes | 5,042 | 11,304 | The generated program and its observable intermediate events increased serialized payload volume. |
| Input tokens | 2,080 | 3,108 | More continuations carried more input context. |
| Output tokens | 385 | 850 | Program generation and intermediate control output increased generated tokens. |
| Reasoning tokens | 119 | 265 | Programmatic used 146 more reasoning tokens (122.7% more). |
| End-to-end latency | 10.715 s | 14.524 s | Programmatic was 3.809 seconds slower in this single run. |
| Estimated cost | $0.023922 | $0.038101 | Programmatic cost $0.014179 more for this run. |

The important conclusion is not that one orchestration style is always cheaper. In this fixed fan-out inventory case, Direct calling was more efficient because all nine independent calls could be requested together. Programmatic Tool Calling is more likely to justify its overhead when tool selection, branching, looping, or in-program aggregation avoids returning large intermediate results through repeated host-managed turns.

#### Medium dataset run

The interpretation below is based on the saved outputs from the `medium` dataset run.

Reference outputs: live comparison trace export and normalized semantic timeline.

Both arms passed the quality gate and executed the same 30 business-tool calls, so their resource measurements are comparable.

| measure | Direct | Programmatic | observation |
| --- | --- | --- | --- |
| Model requests / host round trips | 2 | 3 | Programmatic required one additional Responses API continuation in this run. |
| Intermediate payload bytes | 16,805 | 25,301 | The generated program and observable intermediate events added 8,496 serialized bytes. |
| Input tokens | 4,578 | 3,158 | Programmatic used 1,420 fewer input tokens (31.0% less). |
| Output tokens | 1,061 | 920 | Programmatic used 141 fewer output tokens (13.3% less). |
| Reasoning tokens | 267 | 267 | Both arms used the same number of reasoning tokens. |
| End-to-end latency | 15.873 s | 23.616 s | Programmatic was 7.743 seconds slower in this single run. |
| Estimated cost | $0.059780 | $0.040352 | Programmatic cost $0.019428 less (32.5% lower). |

In this `medium` fixed-fan-out run, Programmatic Tool Calling traded latency and observable payload volume for lower model-token usage and estimated cost. The program coordinated the 30 business-tool calls and reduced their detailed results before the final model response, which more than offset the extra continuation and program-generation overhead in billed tokens. Direct calling remained faster because it requested all independent calls together and needed only one continuation.

#### Why the cost direction changes between the saved runs

First, the direction is the opposite of what the question suggests when we look at the saved results.

| Dataset | Direct | Programmatic | Result |
| --- | --- | --- | --- |
| Small | $0.023922 | $0.038101 | Direct was $0.014179 cheaper. |
| Medium | $0.059780 | $0.040352 | Programmatic was $0.019428 cheaper. |

This cost reversal was caused less by dataset size alone than by differences in how the generated Programmatic execution handled tool calls and continuations in the two saved runs.

##### Why Direct was cheaper for the small dataset

For the `small` run, both approaches executed the same nine tool calls, but their resource usage differed as follows.

| Metric | Direct | Programmatic |
| --- | --- | --- |
| Model requests | 2 | 5 |
| Input tokens | 2,080 | 3,108 |
| Output tokens | 385 | 850 |
| Reasoning tokens | 119 | 265 |

The timeline shows that Programmatic made the inventory and demand calls in the first request, then handled the inbound-shipment calls one at a time across three subsequent requests, and finally generated the answer in a fifth request.

In other words, the following fixed overheads were large relative to the size of the task:

- generating the program;
- making three additional continuations;
- processing additional input tokens to carry forward the prior execution state; and
- generating Programmatic control output and reasoning tokens.

In the pricing snapshot used by this notebook, output tokens cost six times as much as uncached input tokens. Reasoning tokens are included in `output_tokens` and are billed at the output-token rate, so the 465 additional output tokens used by Programmatic, including 146 additional reasoning tokens, had a substantial effect on cost.

##### Why Programmatic was cheaper for the medium dataset

For the `medium` run, both approaches executed 30 tool calls, but the result changed.

| Metric | Direct | Programmatic |
| --- | --- | --- |
| Model requests | 2 | 3 |
| Input tokens | 4,578 | 3,158 |
| Output tokens | 1,061 | 920 |
| Reasoning tokens | 267 | 267 |

Direct returned the detailed results of all 30 tool calls to the next model request, where the model synthesized them into the final answer. As the dataset grew, both input and output token usage increased substantially.

By contrast, the Programmatic execution for the `medium` run:

1. executed 20 inventory and demand calls in the first request;
2. executed all ten inbound-shipment calls together in the second request; and
3. generated the final answer from the aggregated result in the third request.

Unlike the `small` run, where the inbound calls were processed one at a time, all ten inbound calls were grouped into a single continuation. As a result, despite the cost of generating the program, Programmatic used:

- 1,420 fewer input tokens;
- 141 fewer output tokens; and
- the same number of reasoning tokens.

Those token savings were large enough to outweigh Programmatic Tool Calling's fixed overhead.

##### How to interpret the result accurately

Two effects appeared together in these saved runs:

- **Scale effect:** As the dataset grows, the cost of returning every tool result to the model for Direct Tool Calling and asking the model to process the full set grows quickly.
- **Execution-shape effect:** The saved `small` Programmatic run required five model requests, whereas the saved `medium` Programmatic run required only three. The generated programs used different batching patterns.

The current results therefore do not establish that Programmatic Tool Calling will always become cheaper above a particular dataset size. The structure of the generated program and its batching behavior can vary between executions. To identify a reliable crossover point, repeat both arms with isolated cache keys and compare the mean and variance of request count, token usage, cost, latency, and quality pass rate.

Also, `intermediate_payload_bytes` measures the size of locally serialized observable data, not billable API tokens. This is why Programmatic can have a larger payload in the `medium` run while still producing a lower estimated cost.

Evidence files:

- Small comparison
- Small timeline
- Medium comparison
- Medium timeline

### Column and export caveats

- `duration_ms` is useful primarily on `model_request` and tool-execution rows; zero or near-zero values on semantic marker rows are expected.
- `payload_bytes` measures locally serialized observable event data, not billed tokens. Use the API usage fields and pricing calculation for cost comparisons.
- `call_id` identifies an individual program or function call; `caller_id` links a Programmatic function call to the program that issued it.
- The saved 403 `zdr_forbidden` message affects OpenAI Dashboard trace ingestion only. The local normalized timeline remains valid because it is recorded independently of trace export.


## Exercise

Implement a quality-adjusted comparison: return the Programmatic-minus-Direct delta for a metric only when both arms pass. Try `host_round_trips`, `intermediate_payload_bytes`, and `estimated_cost_usd`.


In [ ]:
def quality_adjusted_delta(rows: list[dict[str, object]], metric: str) -> float | None:
    by_arm = {str(row["arm"]): row for row in rows}
    if set(by_arm) != {"direct", "programmatic"}:
        return None
    if not all(bool(row["quality_passed"]) for row in by_arm.values()):
        return None
    return float(by_arm["programmatic"][metric]) - float(by_arm["direct"][metric])

quality_adjusted_delta(trace_metric_rows, "host_round_trips")


## Pitfalls and extensions

- **Common mistake:** Treating a trace as hidden chain-of-thought. The trace contains observable requests, response items, and tool execution spans only.
- **Sensitive data:** Full payloads are retained locally only. Set `INCLUDE_LOCAL_PAYLOADS=False` before adapting this code to real data; exported Trace spans always contain bounded summaries.
- **Trace delivery:** Export is buffered; the adapter flushes after the parent trace closes, but Dashboard appearance can still be delayed.
- **Optional extension:** Add an incident trace where each result can change the next tool decision, then compare why Direct calling may be the better orchestration shape.

Official reference: [Programmatic Tool Calling](https://developers.openai.com/api/docs/guides/tools-programmatic-tool-calling)
